In [3]:
# 03_baseline_models.ipynb

# %% [markdown]
# # Regulatory-Constrained Fairness Framework for Credit Scoring
# ## 03. Baseline Models
#
# Train and compare baseline classifiers on each credit dataset:
# - Logistic Regression, XGBoost, LightGBM, CatBoost
# - class imbalance handled via class weights / scale_pos_weight (no resampling)
# - evaluate with AUC, PR-AUC, F1, balanced accuracy, Brier score
# - persist fitted predictions (test-set scores) for later fairness / RCFS notebooks

# %%
# Imports and configuration
import os
import json
import warnings
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    balanced_accuracy_score, brier_score_loss,
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

DATA_DIR = os.path.join("..", "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
RESULTS_DIR = os.path.join("..", "results")
TAB_DIR = os.path.join(RESULTS_DIR, "tables")

RANDOM_STATE = 42
os.makedirs(TAB_DIR, exist_ok=True)

# %%
# Load metadata and processed splits
with open(os.path.join(PROCESSED_DIR, "preprocess_meta.json")) as f:
    META = json.load(f)

DATASETS = {
    "German Credit": "german",
    "Taiwan CC Default": "taiwan",
    "Credit Card Approval": "credit_card_approval",
}

def load_split(key):
    train = pd.read_parquet(os.path.join(PROCESSED_DIR, f"{key}_train.parquet"))
    test = pd.read_parquet(os.path.join(PROCESSED_DIR, f"{key}_test.parquet"))
    return train, test

# %%
# Split a processed frame into feature matrix X and label y.
# Sensitive columns and any auxiliary target variants are excluded from X.
def get_Xy(df, key):
    meta = META[key]
    target_col = meta["target_col"]
    sensitive_cols = meta["sensitive_cols"]
    aux_cols = [c for c in ["target_60dpd", "target_30dpd"] if c in df.columns and c != target_col]
    drop = [target_col] + sensitive_cols + aux_cols
    X = df.drop(columns=[c for c in drop if c in df.columns], errors="ignore")
    y = df[target_col].astype(int)
    return X, y

# %%
# Model factory. Class imbalance is addressed with class weights / scale_pos_weight.
def build_models(y_train):
    n_pos = int(y_train.sum())
    n_neg = int((y_train == 0).sum())
    spw = (n_neg / n_pos) if n_pos > 0 else 1.0  # scale_pos_weight for boosting

    return {
        "LogisticRegression": LogisticRegression(
            max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE
        ),
        "XGBoost": XGBClassifier(
            n_estimators=400, max_depth=5, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9, scale_pos_weight=spw,
            eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1,
        ),
        "LightGBM": LGBMClassifier(
            n_estimators=400, max_depth=-1, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9, class_weight="balanced",
            random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
        ),
        "CatBoost": CatBoostClassifier(
            iterations=400, depth=5, learning_rate=0.05,
            auto_class_weights="Balanced", random_seed=RANDOM_STATE, verbose=False,
        ),
    }

# %%
# Evaluation helper
def evaluate(y_true, proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    return {
        "AUC": roc_auc_score(y_true, proba),
        "PR_AUC": average_precision_score(y_true, proba),
        "F1": f1_score(y_true, pred, zero_division=0),
        "BalancedAcc": balanced_accuracy_score(y_true, pred),
        "Brier": brier_score_loss(y_true, proba),
    }

# %%
# Train all models on all datasets; collect metrics and test-set probabilities
results = []
predictions = {}  # {dataset_key: {model_name: DataFrame(y_true, proba, + sensitive)}}

for display_name, key in DATASETS.items():
    train, test = load_split(key)
    X_train, y_train = get_Xy(train, key)
    X_test, y_test = get_Xy(test, key)

    # Ensure identical column order
    X_test = X_test[X_train.columns]

    models = build_models(y_train)
    predictions[key] = {}

    for model_name, model in models.items():
        model.fit(X_train, y_train)
        proba = model.predict_proba(X_test)[:, 1]

        metrics = evaluate(y_test.values, proba)
        metrics.update({"dataset": display_name, "model": model_name})
        results.append(metrics)

        # Store predictions with sensitive attributes for downstream fairness analysis
        pred_df = test[META[key]["sensitive_cols"]].copy()
        pred_df["y_true"] = y_test.values
        pred_df["proba"] = proba
        predictions[key][model_name] = pred_df

        print(f"[{display_name:>22}] {model_name:<18} "
              f"AUC={metrics['AUC']:.3f}  PR-AUC={metrics['PR_AUC']:.3f}  F1={metrics['F1']:.3f}")

# %%
# Results table (ordered)
res_df = pd.DataFrame(results)[
    ["dataset", "model", "AUC", "PR_AUC", "F1", "BalancedAcc", "Brier"]
].round(4)
print(res_df.to_string(index=False))
res_df.to_csv(os.path.join(TAB_DIR, "03_baseline_metrics.csv"), index=False)

# %%
# Persist test-set predictions for the fairness / RCFS notebooks
PRED_DIR = os.path.join(PROCESSED_DIR, "predictions")
os.makedirs(PRED_DIR, exist_ok=True)
for key, model_preds in predictions.items():
    for model_name, pred_df in model_preds.items():
        fname = f"{key}__{model_name}.parquet"
        pred_df.to_parquet(os.path.join(PRED_DIR, fname))
print("Saved predictions to:", PRED_DIR)

[         German Credit] LogisticRegression AUC=0.809  PR-AUC=0.636  F1=0.641
[         German Credit] XGBoost            AUC=0.803  PR-AUC=0.636  F1=0.608
[         German Credit] LightGBM           AUC=0.803  PR-AUC=0.649  F1=0.539
[         German Credit] CatBoost           AUC=0.815  PR-AUC=0.675  F1=0.662
[     Taiwan CC Default] LogisticRegression AUC=0.717  PR-AUC=0.499  F1=0.471
[     Taiwan CC Default] XGBoost            AUC=0.774  PR-AUC=0.550  F1=0.533
[     Taiwan CC Default] LightGBM           AUC=0.771  PR-AUC=0.545  F1=0.528
[     Taiwan CC Default] CatBoost           AUC=0.777  PR-AUC=0.553  F1=0.539
[  Credit Card Approval] LogisticRegression AUC=0.544  PR-AUC=0.145  F1=0.210
[  Credit Card Approval] XGBoost            AUC=0.701  PR-AUC=0.285  F1=0.325
[  Credit Card Approval] LightGBM           AUC=0.730  PR-AUC=0.318  F1=0.354
[  Credit Card Approval] CatBoost           AUC=0.660  PR-AUC=0.250  F1=0.284
             dataset              model    AUC  PR_AUC     F1  B